In [ ]:
## offsets in normal direction
## Fusion of XY, YZ, and XZ slicing with axis colors
## choose if x y or z planes should be used for slicing
## gcode object is X,Y=0,0 centered
## meta data as comment in gcode header
## angled slicing planes for all directions


import matplotlib.pyplot as plt
from skimage import measure
import numpy as np
import fullcontrol as fc
from datetime import datetime
import os



# =============================
# PARAMETERS
# =============================
gcode_output = False
plot_output = True

do_XY = True
do_XZ = True
do_YZ = True

use_angled_XY = False
use_angled_XZ = False
use_angled_YZ = False

angle_deg_XY = 45
angle_deg_XZ= 45
angle_deg_YZ = 45

wall_t = 0.25     # base half-thickness between walls
n_walls = 3     # total number of walls per slice
uc_size_mm = 5.0
n_uc_x = 5
n_uc_y = 4
n_uc_z = 3

Lx = n_uc_x * uc_size_mm
Ly = n_uc_y * uc_size_mm
Lz = n_uc_z * uc_size_mm

spacing = 0.3
Z_SHIFT = Lz / 2
Y_SHIFT = Ly / 2
X_SHIFT = Lx / 2


n_slices_x = int(np.ceil(Lx / spacing)) + 1  
n_slices_y = int(np.ceil(Ly / spacing)) + 1
n_slices_z = int(np.ceil(Lz / spacing)) + 1

kx = 2 * np.pi / uc_size_mm
ky = 2 * np.pi / uc_size_mm
kz = 2 * np.pi / uc_size_mm

samples_per_uc = 60

res_x = int(samples_per_uc * n_uc_x)
res_y = int(samples_per_uc * n_uc_y)
res_z = int(samples_per_uc * n_uc_z)

surface_type = "schwarzP"  # options: "schwarzP", "gyroid", "diamond", "fischerkoch", "iwp", "splitp"
color_type = "CMY" # options: "BAM", "CMY"

print(f"Generation of {surface_type} structure has started!\n Size (X, Y, Z): {uc_size_mm*n_uc_x} x {uc_size_mm*n_uc_y} x {uc_size_mm*n_uc_z} \n Wall thickness: {wall_t} \n Spacing: {spacing}")

# Axis colors
if color_type == "BAM":
    COLORS = {
        'XY': (210/255,0,30/255),
        'YZ': (0,175/255,240/255),
        'XZ': (0,40/255,50/255)
    }
elif color_type == "CMY":
    COLORS = {
    'XY': (1,1,0),
    'YZ': (0,1,1),
    'XZ': (1,0,1)
    } 
else:
    raise ValueError(f"Unknown color type: {color_type}")


DEFAULT_COLOR = (0.5, 0.5, 0.5)

# =============================
# SURFACE FUNCTION
# =============================
def surface_func(x, y, z):
    if surface_type == "schwarzP":
        return np.cos(kx*x) + np.cos(ky*y) + np.cos(kz*z)
    elif surface_type == "gyroid":
        return (np.sin(kx*x)*np.cos(ky*y) +
                np.sin(ky*y)*np.cos(kz*z) +
                np.sin(kz*z)*np.cos(kx*x))
    elif surface_type == "diamond":
        return (
            np.sin(kx*x)*np.sin(ky*y)*np.sin(kz*z) +
            np.sin(kx*x)*np.cos(ky*y)*np.cos(kz*z) +
            np.cos(kx*x)*np.sin(ky*y)*np.cos(kz*z) +
            np.cos(kx*x)*np.cos(ky*y)*np.sin(kz*z)
        )
    elif surface_type == "iwp":
        return (
            2*(np.cos(kx*x)*np.cos(ky*y) +
            np.cos(ky*y)*np.cos(kz*z) +
            np.cos(kz*z)*np.cos(kx*x)) -
            np.cos(2*kx*x)-np.cos(2*ky*y)-np.cos(2*kz*z)
        )
    elif surface_type == "splitp":
        return (
            1.1 * (
                np.sin(2*kx*x)*np.sin(kz*z)*np.cos(ky*y) +
                np.sin(2*ky*y)*np.sin(kx*x)*np.cos(kz*z) +
                np.sin(2*kz*z)*np.sin(ky*y)*np.cos(kx*x)
            )
            - 0.2 * (
                np.cos(2*kx*x)*np.cos(2*ky*y) +
                np.cos(2*ky*y)*np.cos(2*kz*z) +
                np.cos(2*kz*z)*np.cos(2*kx*x)
            )
            - 0.4 * (
                np.cos(2*kx*x) +
                np.cos(2*ky*y) +
                np.cos(2*kz*z)
            )
        )

    else:
        raise ValueError(f"Unknown surface type: {surface_type}")

# =============================
# NORMAL / GRADIENT FUNCTION
# =============================
def surface_normal(x, y, z):
    if surface_type == "schwarzP":
        nx = -kx * np.sin(kx*x)
        ny = -ky * np.sin(ky*y)
        nz = -kz * np.sin(kz*z)
    elif surface_type == "gyroid":
        nx = kx*np.cos(kx*x)*np.cos(ky*y) - kz*np.sin(kz*z)*np.sin(kx*x)
        ny = ky*np.cos(ky*y)*np.cos(kz*z) - kx*np.sin(kx*x)*np.sin(ky*y)
        nz = kz*np.cos(kz*z)*np.cos(kx*x) - ky*np.sin(ky*y)*np.sin(kz*z)
    elif surface_type == "diamond":
        sx = np.sin(kx*x)
        cx = np.cos(kx*x)
        sy = np.sin(ky*y)
        cy = np.cos(ky*y)
        sz = np.sin(kz*z)
        cz = np.cos(kz*z)

        nx = kx * (
            cx*sy*sz +
            cx*cy*cz -
            sx*sy*cz -
            sx*cy*sz
        )

        ny = ky * (
            sx*cy*sz -
            sx*sy*cz +
            cx*cy*cz -
            cx*sy*sz
        )

        nz = kz * (
            sx*sy*cz -
            sx*cy*sz +
            cx*sy*cz +
            cx*cy*sz
        )
    elif surface_type == "iwp":

        nx = (
            -2*kx*np.sin(kx*x)*np.cos(ky*y)
            -2*kx*np.sin(kx*x)*np.cos(kz*z)
            +2*kx*np.sin(2*kx*x)
        )

        ny = (
            -2*ky*np.sin(ky*y)*np.cos(kx*x)
            -2*ky*np.sin(ky*y)*np.cos(kz*z)
            +2*ky*np.sin(2*ky*y)
        )

        nz = (
            -2*kz*np.sin(kz*z)*np.cos(kx*x)
            -2*kz*np.sin(kz*z)*np.cos(ky*y)
            +2*kz*np.sin(2*kz*z)
        )
    elif surface_type == "splitp":

        sx = np.sin(kx*x); cx = np.cos(kx*x)
        sy = np.sin(ky*y); cy = np.cos(ky*y)
        sz = np.sin(kz*z); cz = np.cos(kz*z)

        s2x = np.sin(2*kx*x); c2x = np.cos(2*kx*x)
        s2y = np.sin(2*ky*y); c2y = np.cos(2*ky*y)
        s2z = np.sin(2*kz*z); c2z = np.cos(2*kz*z)

        nx = (
            1.1 * (
                2*kx*c2x*sz*cy +
                kx*s2y*cx*cz -
                kx*s2z*sy*sx
            )
            + 0.4*kx*s2x*(c2y + c2z)
            + 0.8*kx*s2x
        )

        ny = (
            1.1 * (
                -ky*s2x*sz*sy +
                2*ky*c2y*sx*cz +
                ky*s2z*cy*cx
            )
            + 0.4*ky*s2y*(c2x + c2z)
            + 0.8*ky*s2y
        )

        nz = (
            1.1 * (
                ky*0 + kx*0 +
                kz*s2x*cz*cy -
                kz*s2y*sx*sz +
                2*kz*c2z*sy*cx
            )
            + 0.4*kz*s2z*(c2x + c2y)
            + 0.8*kz*s2z
        )

    else:
        raise ValueError(f"Unknown surface type: {surface_type}")

    n = np.array([nx, ny, nz], dtype=float)
    norm = np.linalg.norm(n)
    if norm == 0:
        return np.array([0,0,1])
    return n / norm

# =============================
# LATITUDE CONTOURS FUNCTIONS
# =============================

def latitudes(plane, offset, angled=False, normal=None):
    """
    Generate TPMS contour loops.

    Parameters
    ----------
    plane : {"xy", "yz", "xz"}
        Orientation of the slicing plane.

    offset : float
        If angled=False:
            z0 (xy), x0 (yz), or y0 (xz).

        If angled=True:
            Signed distance from cube center along the plane normal.

    angled : bool
        False -> axis-aligned slicing
        True  -> arbitrary plane with given normal

    normal : (3,) array_like
        Plane normal (required when angled=True).

    Returns
    -------
    list of (N,3) arrays
    """

    plane_info = {
        "xy": dict(
            res=(res_x, res_y),
            lengths=(Lx, Ly),
            fixed_axis=2,
            tmp=np.array([1,0,0]),
        ),
        "yz": dict(
            res=(res_y, res_z),
            lengths=(Ly, Lz),
            fixed_axis=0,
            tmp=np.array([1,0,0]),
        ),
        "xz": dict(
            res=(res_x, res_z),
            lengths=(Lx, Lz),
            fixed_axis=1,
            tmp=np.array([0,1,0]),
        ),
    }

    info = plane_info[plane]
    nu, nv = info["res"]

    # ==========================================================
    # AXIS-ALIGNED SLICES
    # ==========================================================

    if not angled:

        if plane == "xy":

            x = np.linspace(0, Lx, res_x)
            y = np.linspace(0, Ly, res_y)

            X, Y = np.meshgrid(x, y, indexing="ij")
            Z = np.full_like(X, offset)

        elif plane == "yz":

            y = np.linspace(0, Ly, res_y)
            z = np.linspace(0, Lz, res_z)

            Y, Z = np.meshgrid(y, z, indexing="ij")
            X = np.full_like(Y, offset)

        elif plane == "xz":

            x = np.linspace(0, Lx, res_x)
            z = np.linspace(0, Lz, res_z)

            X, Z = np.meshgrid(x, z, indexing="ij")
            Y = np.full_like(X, offset)

        F = surface_func(X, Y, Z)

        loops = []

        for c in measure.find_contours(F, 0.0):

            if plane == "xy":

                xs = np.interp(c[:,0], np.arange(res_x), x)
                ys = np.interp(c[:,1], np.arange(res_y), y)
                zs = np.full_like(xs, offset)

            elif plane == "yz":

                ys = np.interp(c[:,0], np.arange(res_y), y)
                zs = np.interp(c[:,1], np.arange(res_z), z)
                xs = np.full_like(ys, offset)

            else:  # xz

                xs = np.interp(c[:,0], np.arange(res_x), x)
                zs = np.interp(c[:,1], np.arange(res_z), z)
                ys = np.full_like(xs, offset)

            loop = np.vstack([xs, ys, zs]).T

            if len(loop) > 20:
                loops.append(loop)

        return loops

    # ==========================================================
    # ANGLED SLICES
    # ==========================================================

    if normal is None:
        raise ValueError("normal must be supplied for angled slices.")

    normal = np.asarray(normal, dtype=float)
    normal /= np.linalg.norm(normal)

    tmp = info["tmp"]

    if np.linalg.norm(np.cross(normal, tmp)) < 1e-8:
        tmp = np.array([0,0,1])

    e1 = np.cross(normal, tmp)
    e1 /= np.linalg.norm(e1)

    e2 = np.cross(normal, e1)

    center = np.array([Lx/2, Ly/2, Lz/2])
    p0 = center + normal * offset

    corners = np.array([
        [0,0,0],
        [Lx,0,0],
        [0,Ly,0],
        [0,0,Lz],
        [Lx,Ly,0],
        [Lx,0,Lz],
        [0,Ly,Lz],
        [Lx,Ly,Lz],
    ])

    rel = corners - p0

    u_proj = rel @ e1
    v_proj = rel @ e2

    u_min, u_max = u_proj.min(), u_proj.max()
    v_min, v_max = v_proj.min(), v_proj.max()

    u = np.linspace(u_min, u_max, nu)
    v = np.linspace(v_min, v_max, nv)

    U, V = np.meshgrid(u, v, indexing="ij")

    X = p0[0] + U*e1[0] + V*e2[0]
    Y = p0[1] + U*e1[1] + V*e2[1]
    Z = p0[2] + U*e1[2] + V*e2[2]

    mask = (
        (X >= 0) & (X <= Lx) &
        (Y >= 0) & (Y <= Ly) &
        (Z >= 0) & (Z <= Lz)
    )

    F = np.full_like(X, np.nan)
    F[mask] = surface_func(X[mask], Y[mask], Z[mask])

    loops = []

    for c in measure.find_contours(F, 0.0):

        us = np.interp(c[:,0], np.arange(nu), u)
        vs = np.interp(c[:,1], np.arange(nv), v)

        xs = p0[0] + us*e1[0] + vs*e2[0]
        ys = p0[1] + us*e1[1] + vs*e2[1]
        zs = p0[2] + us*e1[2] + vs*e2[2]

        inside = (
            (xs >= 0) & (xs <= Lx) &
            (ys >= 0) & (ys <= Ly) &
            (zs >= 0) & (zs <= Lz)
        )

        loop = np.vstack([
            xs[inside],
            ys[inside],
            zs[inside]
        ]).T

        if len(loop) > 20:
            loops.append(loop)

    return loops

# =============================
# NORMAL-BASED MULTI-WALL OFFSET
# =============================
def thicken_path_normal_multi(path, wall_t, n_walls):

    shells = []

    # correct offsets for even + odd
    center = (n_walls - 1) / 2

    offsets = [
        (i - center) * wall_t
        for i in range(n_walls)
    ]

    for offset in offsets:

        shell = []

        for p in path:

            n = surface_normal(*p)

            shell.append([
                p[0] + offset * n[0],
                p[1] + offset * n[1],
                p[2] + offset * n[2]
            ])

        shells.append(np.array(shell))

    return shells

# =============================
# GENERATE ALL LOOPS (XY + YZ + XZ)
# =============================
all_loops = []

# XY slices
if do_XY:

    if use_angled_XY:

        a = np.deg2rad(angle_deg_XY)

        normal = np.array([
            np.sin(a),
            0,
            np.cos(a)
        ])

        normal /= np.linalg.norm(normal)

        # max distance through cube

        max_d = np.linalg.norm([Lx, Ly, Lz])

        d_values = np.arange(-max_d/2, max_d/2, spacing)

        for d in d_values:

            loops = latitudes("xy", d, angled=True, normal=normal)

            for loop in loops:
                all_loops.append((loop, 'XY'))

    else:

        z_values = np.linspace(0, Lz, n_slices_z)

        for z0 in z_values:
            for loop in latitudes("xy", z0):
                all_loops.append((loop, 'XY'))

# YZ slices
if do_YZ:
    if use_angled_YZ:
        a = np.deg2rad(angle_deg_YZ)

        normal = np.array([0, np.cos(a), np.sin(a)])

        normal /= np.linalg.norm(normal)

        # max distance through cube

        max_d = np.linalg.norm([Lx, Ly, Lz])

        d_values = np.arange(-max_d/2, max_d/2, spacing)

        for d in d_values:

            loops = latitudes("yz", d, angled=True, normal=normal)

            for loop in loops:
                all_loops.append((loop, 'YZ'))

    else:
        x_values = np.linspace(0, Lx, n_slices_x)
        for x0 in x_values:
            for loop in latitudes("yz", x0):
                all_loops.append((loop, 'YZ'))

# XZ slices
if do_XZ:
    if use_angled_XZ:
        a = np.deg2rad(angle_deg_XZ)

        normal = np.array([np.cos(a), np.sin(a), 0])

        normal /= np.linalg.norm(normal)

        # max distance through cube

        max_d = np.linalg.norm([Lx, Ly, Lz])

        d_values = np.arange(-max_d/2, max_d/2, spacing)

        for d in d_values:

            loops = latitudes("xz", d, angled=True, normal=normal)

            for loop in loops:
                all_loops.append((loop, 'XZ'))

    else:
        y_values = np.linspace(0, Ly, n_slices_y)
        for y0 in y_values:
            for loop in latitudes("xz", y0):
                all_loops.append((loop, 'XZ'))

print(f"Generated total {len(all_loops)} loops (XY + YZ + XZ)")

# =============================
# FULLCONTROL STEPS
# =============================
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d %H:%M:%S")

# GCODE comment with meta data
steps = []
steps.append(fc.GcodeComment(text=f"###### MANUAL GCODE COMMENT START ######"))
## steps.append(fc.GcodeComment(text=f"Version of Script: {os.path.basename(__file__)}"))
steps.append(fc.GcodeComment(text=f"User: {os.getlogin()}"))
steps.append(fc.GcodeComment(text=f"Timestamp: {timestamp}"))
steps.append(fc.GcodeComment(text=f"Surface Type = {surface_type}"))
steps.append(fc.GcodeComment(text=f"Unit Cell Size = {uc_size_mm} mm"))
steps.append(fc.GcodeComment(text=f"Number of Walls = {n_walls}"))
steps.append(fc.GcodeComment(text=f"Wall offset = {wall_t} mm"))
steps.append(fc.GcodeComment(text=f"Spacing = {spacing} mm"))
steps.append(fc.GcodeComment(text=f"Number of Unit Cells X = {n_uc_x}"))
steps.append(fc.GcodeComment(text=f"Number of Unit Cells Y = {n_uc_y}"))
steps.append(fc.GcodeComment(text=f"Number of Unit Cells Z = {n_uc_z}"))
steps.append(fc.GcodeComment(text=f"Total structure size (X, Y, Z) {uc_size_mm*n_uc_x} x {uc_size_mm*n_uc_y} x {uc_size_mm*n_uc_z}"))
steps.append(fc.GcodeComment(text=f"###### MANUAL GCODE COMMENT END ######"))



steps.append(fc.Extruder(on=False))
steps.append(fc.Point(x=0, y=0, z=0, color=DEFAULT_COLOR))

for path, axis in all_loops:
    shells = thicken_path_normal_multi(path, wall_t, n_walls)
    color = COLORS[axis] 
    for shell in shells:
        p0 = shell[0]
        steps.append(fc.Point(x=float(p0[0] - X_SHIFT), y=float(p0[1] - Y_SHIFT), z=float(p0[2]), color=color))
        steps.append(fc.Extruder(on=True))
        for p in shell[1:]:
            steps.append(fc.Point(x=float(p[0] - X_SHIFT), y=float(p[1] - Y_SHIFT), z=float(p[2]), color=color))
        steps.append(fc.Extruder(on=False))

# Retract & home
steps.append(fc.Point(x=0, y=0, z=Z_SHIFT, color=DEFAULT_COLOR))
steps.append(fc.Point(x=0, y=0, z=0, color=DEFAULT_COLOR))

print(f"Generated {surface_type} structure with size (X, Y, Z) {uc_size_mm*n_uc_x} x {uc_size_mm*n_uc_y} x {uc_size_mm*n_uc_z}")

# =============================
# PLOT PATHS
# =============================
if plot_output == True:
    print("Plotting...")
    fig = fc.transform(
        steps,
        'plot',
        fc.PlotControls(
            initialization_data={'extrusion_width':0.2,'extrusion_height':0.2},
            hide_travel=True,
            style='line',
            zoom=0.5,
            color_type='manual',
            neat_for_publishing=True
        )
    )


    fig.update_layout(
        template='plotly_white',
        paper_bgcolor='gray',
        font=dict(family='Arial', size=14, color='black'),
        scene=dict(
            bgcolor='gray',
            xaxis=dict(visible=False, showgrid=True, zeroline=False),
            yaxis=dict(visible=False, showgrid=False, zeroline=False),
            zaxis=dict(visible=False, showgrid=False, zeroline=False),
        ),
    )
    fig.show()

# =============================
# OPTIONAL GCODE OUTPUT
# =============================

if gcode_output == True:
    save_as = f"output_{surface_type}_{n_uc_x}_x_{n_uc_x}_x_{n_uc_z}_UCS_{uc_size_mm}"
    gcode_controls = fc.GcodeControls(save_as=save_as, printer_name='generic') # filename includes date+time
    fc.transform(steps, 'gcode', gcode_controls, show_tips=False)
    print("gcode exported...")

